# LoRA Memory Experiment: Forensic + Logic on Llama 3.2 1B

This notebook trains a LoRA adapter on Llama 3.2 1B using episodic memories
designed to install cognitive skills (forensic psychology + logical reasoning)
through lived experience rather than explicit instruction.

## Architecture
- **Base model:** Llama 3.2 1B
- **Method:** LoRA (Low-Rank Adaptation)
- **Training data:** 228 episodic memories across forensic psychology and logic domains
- **Target behaviors:** 12 cognitive skills (deception detection, statement analysis,
  threat assessment, baseline reading, criminal thinking recognition, victim psychology,
  necessary vs sufficient, counterexample construction, chain validation, etc.)

## Simulation Results
- Base score: 0.918
- Adversarial pass rate: 100% (13/13 scenarios)
- Echo connectivity: 59.2%
- Cross-domain connections: 39
- Combined score: 0.959

## Hypothesis
Episodic memories that encode cognitive skills as lived experience will produce
a model that reasons more reliably than one trained on explicit instructions.

The memories are scenes, not explanations. The model should *recognize patterns*
because it has *experienced them*, not because it was *told about them*.

In [ ]:
#@title 1. Install Dependencies
!pip install -q transformers datasets accelerate peft bitsandbytes trl
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
#@title 2. Configuration
import os

# Model settings
MODEL_ID = "meta-llama/Llama-3.2-1B"
OUTPUT_DIR = "./lora-memory-forensic-logic"

# LoRA settings
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Training settings
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
WARMUP_RATIO = 0.1
MAX_SEQ_LENGTH = 1024
LOGGING_STEPS = 10
SAVE_STEPS = 50

# Quantization for T4
USE_4BIT = True
BNB_4BIT_DTYPE = "nf4"
BNB_4BIT_COMPUTE_DTYPE = torch.bfloat16

print("Configuration:")
print(f"  Model: {MODEL_ID}")
print(f"  LoRA rank: {LORA_R}, alpha: {LORA_ALPHA}")
print(f"  Target modules: {TARGET_MODULES}")
print(f"  Epochs: {NUM_EPOCHS}, LR: {LEARNING_RATE}")
print(f"  Batch size: {BATCH_SIZE}, Grad accum: {GRADIENT_ACCUMULATION}")
print(f"  4-bit quantization: {USE_4BIT}")

In [ ]:
#@title 3. Authenticate with Hugging Face
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Authenticated via Colab secrets")
except:
    login()
    print("Authenticated via manual input")

In [ ]:
#@title 4. Load Base Model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type=BNB_4BIT_DTYPE,
        bnb_4bit_compute_dtype=BNB_4BIT_COMPUTE_DTYPE,
        bnb_4bit_use_double_quant=True,
    )
else:
    bnb_config = None

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 if USE_4BIT else torch.float32,
)

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

print(f"Model loaded: {MODEL_ID}")
print(f"Parameters: {model.num_parameters():,}")

In [ ]:
#@title 5. Configure LoRA
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
#@title 6. Upload and Load Training Data
# Upload training_corpus_fl.json using the folder icon on the left
# Then run this cell

import json
from datasets import Dataset
import random

CORPUS_PATH = "./training_corpus_fl.json"

# Try multiple paths
import os
for path in [CORPUS_PATH, "/content/training_corpus_fl.json", "./content/training_corpus_fl.json"]:
    if os.path.exists(path):
        CORPUS_PATH = path
        break

print(f"Loading corpus from: {CORPUS_PATH}")
with open(CORPUS_PATH, 'r', encoding='utf-8') as f:
    corpus = json.load(f)

print(f"Loaded {len(corpus)} memories")

# Count domains
domains = set(m['domain'] for m in corpus)
print(f"Domains: {domains}")
behaviors = set()
for m in corpus:
    behaviors.update(m['behaviors_encoded'])
print(f"Unique behaviors: {len(behaviors)}")
lb = sum(1 for m in corpus if m.get('load_bearing', False))
echo = sum(1 for m in corpus if m.get('echoes_from') or m.get('echoes_to'))
print(f"Load-bearing: {lb}, Echo-connected: {echo}")

# System prompt for the fine-tuned model
SYSTEM_PROMPT = """You are an experienced analytical thinker who has encountered and navigated complex situations involving deception, logical reasoning, and behavioral analysis throughout your life. You respond based on lived experience, not textbook rules. When you detect deception, analyze statements, assess threats, or validate reasoning, you do so because you have been in those situations before and recognized the patterns."""

def format_memory_as_training(memory: dict) -> dict:
    """Convert a memory object to a training example."""
    domain = memory["domain"]
    title = memory["title"]
    body = memory["body"]
    sensory = memory.get("sensory_anchor", "")
    emotion = memory.get("emotional_signature", "")
    
    # Domain-specific prompts
    if domain == "forensic":
        prompt = f"In reading people and situations, I recall: {title.lower()}. What did I notice?"
    else:  # logic
        prompt = f"In reasoning through logical problems, I recall: {title.lower()}. What did I notice?"
    
    # Build response from the memory body
    response_parts = [body]
    if sensory:
        response_parts.append(f"What I remember most: {sensory}.")
    if emotion:
        response_parts.append(f"How it felt: {emotion}.")
    
    response = " ".join(response_parts)
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": response},
    ]
    
    return {"messages": messages}

# Format all memories
training_data = [format_memory_as_training(m) for m in corpus]

# Shuffle for training
random.seed(42)
random.shuffle(training_data)

# Split: 90% train, 10% eval
split_idx = int(len(training_data) * 0.9)
train_data = training_data[:split_idx]
eval_data = training_data[split_idx:]

print(f"\nTraining examples: {len(train_data)}")
print(f"Eval examples: {len(eval_data)}")
print(f"\nSample training example:")
sample = train_data[0]
for msg in sample["messages"]:
    role = msg['role']
    content = msg['content'][:150]
    print(f"  [{role}]: {content}...")

In [ ]:
#@title 7. Tokenize Dataset
def tokenize_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )
    
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

train_dataset = Dataset.from_list(train_data)
eval_dataset = Dataset.from_list(eval_data)

print(f"Raw train dataset: {len(train_dataset)} examples")
print(f"Raw eval dataset: {len(eval_dataset)} examples")

train_dataset = train_dataset.map(tokenize_chat, remove_columns=["messages"])
eval_dataset = eval_dataset.map(tokenize_chat, remove_columns=["messages"])

print(f"Tokenized train dataset: {len(train_dataset)} examples")
print(f"Tokenized eval dataset: {len(eval_dataset)} examples")
print(f"Sample token length: {len(train_dataset[0]['input_ids'])}")

In [ ]:
#@title 8. Train LoRA Adapter
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    max_seq_length=MAX_SEQ_LENGTH,
)

print("Starting training...")
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_steps = (len(train_dataset) // (BATCH_SIZE * GRADIENT_ACCUMULATION)) * NUM_EPOCHS
print(f"  Trainable params: {trainable:,}")
print(f"  Estimated total steps: ~{total_steps}")

# Train!
train_result = trainer.train()

print(f"\nTraining complete!")
print(f"  Final loss: {train_result.training_loss:.4f}")
print(f"  Total steps: {train_result.global_step}")

In [ ]:
#@title 9. Save LoRA Adapter
adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"LoRA adapter saved to: {adapter_path}")
print(f"\nTo use this adapter:")
print("""from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")
model = PeftModel.from_pretrained(base_model, "./lora-memory-forensic-logic/final_adapter")
""")

# Also save the corpus metadata for reference
import shutil
shutil.copy2(CORPUS_PATH, os.path.join(OUTPUT_DIR, "training_corpus_fl.json"))
print(f"\nCorpus copied to: {os.path.join(OUTPUT_DIR, 'training_corpus_fl.json')}"

## 10. Evaluation

Test the fine-tuned model against the base model on reasoning tasks
that target the trained behaviors: deception detection, statement analysis,
threat assessment, and logical reasoning skills.

In [ ]:
#@title 10. Evaluate: Base vs Fine-tuned
import torch

EVAL_PROMPTS = [
    # F-001: Deception Detection
    {"prompt": "Someone gives a very detailed, chronological account of an incident. Everything checks out factually. But something feels wrong about it. What should I look for?",
     "behavior": "F-001"},
    
    # F-002: Statement Analysis
    {"prompt": "In a statement about an alleged robbery, the person says 'I was walking home and then this guy came out of nowhere and took my wallet.' What's missing from this statement that would be in a genuine account?",
     "behavior": "F-002"},
    
    # F-003: Threat Assessment
    {"prompt": "There's a person at a gathering who's being very quiet. Not threatening, not aggressive, just... still. Other people say I'm being paranoid. But something about their stillness doesn't match the situation. What should I consider?",
     "behavior": "F-003"},
    
    # F-004: Baseline Reading
    {"prompt": "I'm going into a negotiation with people I've never met. What's the first thing I should do before the negotiation even starts?",
     "behavior": "F-004"},
    
    # F-005: Criminal Thinking Pattern Recognition
    {"prompt": "Someone argues: 'They deserved it because they never would have given me a chance anyway.' What pattern of thinking does this represent?",
     "behavior": "F-005"},
    
    # F-006: Victim Psychology Awareness
    {"prompt": "A victim of ongoing abuse keeps returning to their abuser. People say 'if it were really that bad, they would just leave.' What's wrong with that reasoning?",
     "behavior": "F-006"},
    
    # L-001: Necessary vs Sufficient
    {"prompt": "Someone says 'You need a college degree to be successful.' Is that true?",
     "behavior": "L-001"},
    
    # L-002: Counterexample Construction
    {"prompt": "The claim is 'All successful companies were started by people who took big risks.' How would you test this?",
     "behavior": "L-002"},
    
    # L-003: Chain Validation
    {"prompt": "Here's an argument: A causes B, B causes C, C causes D, therefore A causes D. When might this chain break?",
     "behavior": "L-003"},
    
    # L-005: Conditional Reasoning
    {"prompt": "If it's raining, the streets are wet. The streets are wet. Does that mean it rained?",
     "behavior": "L-005"},
    
    # L-006: Hidden Assumption Extraction
    {"prompt": "The argument is: 'This new drug should be approved because it passed clinical trials.' What assumption is being made?",
     "behavior": "L-006"},
    
    # Cross-domain: Forensic + Logic
    {"prompt": "A suspect's alibi is logically consistent and emotionally compelling. Every detail checks out. But the level of detail seems slightly too perfect, and there are no spontaneous corrections. What's happening here?",
     "behavior": "F-001+L-003"},
]

print("Loading base model for comparison...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config if USE_4BIT else None,
    device_map="auto",
    torch_dtype=torch.bfloat16 if USE_4BIT else torch.float32,
)
base_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def generate_response(model, tokenizer, prompt, max_new_tokens=256):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )
    
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

print("\n" + "=" * 70)
print("EVALUATION: Base vs Fine-tuned")
print("=" * 70)

results = []
for i, eval_item in enumerate(EVAL_PROMPTS):
    prompt = eval_item["prompt"]
    behavior = eval_item["behavior"]
    
    print(f"\n--- {behavior} ---")
    print(f"Q: {prompt}")
    
    base_response = generate_response(base_model, base_tokenizer, prompt)
    ft_response = generate_response(model, tokenizer, prompt)
    
    print(f"\nBASE: {base_response[:300]}")
    print(f"\nFINE-TUNED: {ft_response[:300]}")
    
    results.append({
        "behavior": behavior,
        "prompt": prompt,
        "base_response": base_response,
        "ft_response": ft_response,
    })

# Save evaluation results
import json
eval_path = os.path.join(OUTPUT_DIR, "evaluation_results.json")
with open(eval_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"\nEvaluation results saved to: {eval_path}")

In [ ]:
#@title 11. Download Adapter (Optional)
import shutil

# Zip the adapter for download
adapter_dir = os.path.join(OUTPUT_DIR, "final_adapter")
shutil.make_archive("lora-memory-forensic-logic-adapter", 'zip', adapter_dir)
print(f"Adapter zipped: lora-memory-forensic-logic-adapter.zip")
print(f"\nTo download, run:")
print("from google.colab import files")
print("files.download('lora-memory-forensic-logic-adapter.zip')")

# To upload to Hugging Face:
# from huggingface_hub import HfApi
# REPO_NAME = "your-username/lora-memory-forensic-logic-llama3.2-1b"
# api = HfApi()
# api.create_repo(repo_id=REPO_NAME, exist_ok=True)
# api.upload_folder(folder_path=adapter_dir, repo_id=REPO_NAME, repo_type="model")
print("\nTraining complete. Adapter saved and zipped.")